# Explore the lakehouse with a SparkSession

This notebook runs **inside the `spark` container** of the POC with the same Spark 3.5.8 + Iceberg 1.11.0 binaries and the same `spark-defaults.conf` as the ETL jobs. It talks to the Lakekeeper REST catalog (`lakehouse`) and the MinIO-compatible storage.

> This is the **platform-team / ETL identity** path (service account), not an analyst path: the notebook runs *as the engine identity*. Analysts never get this notebook in production; they use Trino or Kyuubi (see `docs/security-model.md`).

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName("explore-notebook").master("local[2]")
         .config("spark.driver.memory", "1g").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version, '| default catalog:', spark.conf.get('spark.sql.defaultCatalog'))

## Browse namespaces and tables

In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show()
for ns in ['bronze', 'silver', 'gold']:
    spark.sql(f"SHOW TABLES IN lakehouse.{ns}").show(truncate=False)

## Current state of silver (what analysts see on both engines)

In [ ]:
spark.sql("""
SELECT c.loyalty_tier, COUNT(*) AS txns, ROUND(SUM(t.amount), 2) AS revenue
FROM lakehouse.silver.transactions t JOIN lakehouse.silver.customers c ON t.customer_id = c.customer_id
WHERE t.transaction_date BETWEEN DATE '2026-09-01' AND DATE '2026-09-14' AND t.status = 'completed'
GROUP BY c.loyalty_tier ORDER BY revenue DESC
""").show()

## Snapshots, history and time travel (platform team only)

Analysts are blocked from `.snapshots`/`.history` on both engines (P4); here we are the ETL identity so we can inspect commits and confirm the CDC merge.

In [ ]:
spark.sql("SELECT snapshot_id, committed_at, operation, summary['added-records'] AS added, summary['deleted-records'] AS deleted FROM lakehouse.silver.customers.snapshots ORDER BY committed_at").show(truncate=False)

In [ ]:
# customer 97 was deleted by the CDC delta, 14 updated, 50001 inserted
spark.sql("SELECT customer_id, segment, loyalty_tier, updated_at FROM lakehouse.silver.customers WHERE customer_id IN (1, 14, 97, 50001) ORDER BY customer_id").show()

In [ ]:
snaps = spark.sql("SELECT snapshot_id FROM lakehouse.silver.customers.snapshots ORDER BY committed_at").collect()
first = snaps[0].snapshot_id
print('earliest snapshot:', first)
spark.sql(f"SELECT customer_id, segment FROM lakehouse.silver.customers VERSION AS OF {first} WHERE customer_id = 97").show()

## Data files, partitions and sizes (P9 / P10)

In [ ]:
spark.sql("SELECT partition, record_count, ROUND(file_size_in_bytes/1024/1024, 2) AS mb, file_path FROM lakehouse.silver.transactions.files ORDER BY partition LIMIT 10").show(truncate=60)
spark.sql("SELECT COUNT(*) AS files, ROUND(SUM(file_size_in_bytes)/1024/1024, 1) AS total_mb, ROUND(AVG(file_size_in_bytes)/1024/1024, 2) AS avg_mb FROM lakehouse.silver.transactions.files").show()

## Manual manipulation (ETL identity)

Example: append a correction to bronze and re-run the silver MERGE logic. Uncomment to try; re-run `scripts/seed.sh` afterwards to return to the reproducible state.

In [ ]:
# spark.sql("INSERT INTO lakehouse.bronze.customers_cdc SELECT 'U', current_timestamp(), customer_id, first_name, last_name, 'enterprise', country, city, 'platinum', signup_date, marketing_opt_in, email, phone, national_id, date_of_birth, street_address, 3 FROM lakehouse.bronze.customers_cdc WHERE customer_id = 1 AND op = 'I'")
# import subprocess; print(subprocess.run(['spark-submit', '/workspace/spark/seed_warehouse.py'], capture_output=True, text=True).stdout[-2000:])

## Pandas hand-off for ad-hoc analysis

In [ ]:
pdf = spark.sql("SELECT region, sales_date, SUM(revenue) AS revenue FROM lakehouse.gold.daily_sales_by_store GROUP BY region, sales_date ORDER BY sales_date").toPandas()
pdf.pivot(index='sales_date', columns='region', values='revenue').tail(10)

In [ ]:
spark.stop()